# Task 2 — Full fine-tune Vietnamese Reranker trên RTX PRO 6000 96GB

Notebook chỉ dùng `QA/train.json`, `index_qa` và `emb_qa_v2` của Task 2. 1.000 câu đầu được giữ ngoài train; 6.000 câu còn lại dùng để mine pseudo-label. Model gốc luôn là `AITeamVN/Vietnamese_Reranker`, không dùng checkpoint Task 1.

Cấu hình nặng: full fine-tune 568M tham số, BF16, sequence 384, 8 hard negatives và effective batch 64.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, shutil, subprocess, sys, gzip

REPO_URL = 'https://github.com/caubenq9999/LawRetrieval.git'
BRANCH = 'feat/qa_baseline'
ROOT = Path('/content/LawRetrieval')
DRIVE_ROOT = Path('/content/drive/MyDrive/DSC2026/task2_qa')
INPUT = DRIVE_ROOT / 'input'
CACHE = DRIVE_ROOT / 'artifacts'
QA_DIR = Path('/content/task2_qa')
WORK = Path('/content/task2_work')
CACHE.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(ROOT / 'requirements.txt')], check=True)

import torch
assert torch.cuda.is_available(), 'Runtime chưa bật GPU'
props = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(props.total_memory / 2**30, 1), 'GiB')
assert props.total_memory / 2**30 >= 40, 'Full FT cần >=40GB; GPU nhỏ dùng LoRA'
subprocess.run(['git', '-C', str(ROOT), 'log', '-1', '--oneline'], check=True)
assert (ROOT / 'Finetune-LegalIR/mine_qa_reranker.py').exists(), (
    'Branch chưa có miner Task 2 mới; hãy push code trước.')

## Khôi phục dữ liệu và artifact Task 2 từ Drive
Notebook giả định cache đã được tạo bởi `COLAB_TASK2_QA.ipynb`.

In [ ]:
required = [INPUT / 'train.json', INPUT / 'public-official.json',
            INPUT / 'selected-contexts.zip', CACHE / 'chunks_qa.jsonl',
            CACHE / 'index_qa.zip', CACHE / 'emb_qa_v2.zip']
missing = [str(p) for p in required if not p.exists()]
assert not missing, 'Thiếu trên Drive:\n' + '\n'.join(missing)

QA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(INPUT / 'train.json', QA_DIR / 'train.json')
shutil.copy2(INPUT / 'public-official.json', QA_DIR / 'public-official.json')
context_dir = QA_DIR / 'selected-contexts'
if len(list(context_dir.glob('context_*.json'))) != 8532:
    subprocess.run(['unzip', '-q', '-o', str(INPUT / 'selected-contexts.zip'),
                    '-d', str(QA_DIR)], check=True)

chunks = WORK / 'chunks_qa.jsonl'
index_dir = WORK / 'index_qa'
emb_dir = WORK / 'emb_qa_v2'
if not chunks.exists():
    shutil.copy2(CACHE / 'chunks_qa.jsonl', chunks)
if not (index_dir / 'meta.json').exists():
    subprocess.run(['unzip', '-q', '-o', str(CACHE / 'index_qa.zip'),
                    '-d', str(WORK)], check=True)
if not (emb_dir / 'meta.json').exists():
    subprocess.run(['unzip', '-q', '-o', str(CACHE / 'emb_qa_v2.zip'),
                    '-d', str(WORK)], check=True)

index_meta = json.loads((index_dir / 'meta.json').read_text())
emb_meta = json.loads((emb_dir / 'meta.json').read_text())
assert index_meta['n_chunks'] == 487194, index_meta['n_chunks']
assert emb_meta['n'] == 487194, emb_meta['n']
# Index cache co the ghi absolute path cua runtime cu.
index_meta['chunks_path'] = str(chunks.resolve())
(index_dir / 'meta.json').write_text(json.dumps(index_meta, indent=2), encoding='utf-8')
print('READY:', index_meta['n_chunks'], 'chunks/embedding rows')

## Mine toàn bộ 6.000 câu train với 8 hard negatives
Các câu `0..999` không bao giờ được miner đọc, dùng làm validation sạch.

In [ ]:
PAIRS = WORK / 'pairs_qa_reranker_6k_n8.jsonl'
CACHED_PAIRS = CACHE / 'pairs_qa_reranker_6k_n8.jsonl.gz'
miner = ROOT / 'Finetune-LegalIR/mine_qa_reranker.py'

if not PAIRS.exists():
    if CACHED_PAIRS.exists():
        with gzip.open(CACHED_PAIRS, 'rb') as src, open(PAIRS, 'wb') as dst:
            shutil.copyfileobj(src, dst)
    else:
        subprocess.run([
            sys.executable, '-u', str(miner),
            '--qa-dir', str(QA_DIR), '--index', str(index_dir),
            '--emb', str(emb_dir), '--out', str(PAIRS),
            '--offset', '1000', '--limit', '6000',
            '--alpha', '0.7', '--pool', '2000',
            '--ndocs', '20', '--mchunks', '3', '--neg', '8',
            '--qbatch', '512'
        ], cwd=str(ROOT), check=True)
        with open(PAIRS, 'rb') as src, gzip.open(CACHED_PAIRS, 'wb', compresslevel=5) as dst:
            shutil.copyfileobj(src, dst)
        shutil.copy2(str(PAIRS) + '.meta.json', CACHE / (PAIRS.name + '.meta.json'))

with open(PAIRS, encoding='utf-8') as f:
    pair_count = sum(1 for _ in f)
print('Usable questions:', pair_count)
assert pair_count >= 4000, 'Pseudo-label quá ít; xem file meta trước khi train'

## Smoke test và tự chọn batch
Thử batch 64 trước. Nếu OOM, process con thoát và VRAM được giải phóng trước khi thử 48/32.

In [ ]:
trainer = ROOT / 'Finetune-LegalIR/train_ce.py'
FT_MODEL = WORK / 'qa_reranker_full_e1'
BASE_MODEL = 'AITeamVN/Vietnamese_Reranker'
TRAIN_BATCH = None

for candidate_batch in (64, 48, 32):
    print('\nSMOKE batch =', candidate_batch)
    result = subprocess.run([
        sys.executable, '-u', str(trainer), '--base', BASE_MODEL,
        '--data', str(PAIRS), '--out', str(FT_MODEL),
        '--log', str(WORK / f'smoke_full_b{candidate_batch}.log'),
        '--full', '--precision', 'bf16', '--no-checkpointing',
        '--epochs', '1', '--batch', str(candidate_batch), '--accum', '1',
        '--neg', '8', '--maxlen', '384', '--lr', '2e-5',
        '--workers', '4', '--smoke'
    ], cwd=str(ROOT / 'Finetune-LegalIR'))
    if result.returncode == 0:
        TRAIN_BATCH = candidate_batch
        break
    torch.cuda.empty_cache()

assert TRAIN_BATCH is not None, 'Batch 32 vẫn lỗi; đọc smoke log để biết có phải OOM không'
print('CHOSEN TRAIN_BATCH =', TRAIN_BATCH)

## Full fine-tune — epoch 1
Nếu phải lùi xuống batch 32, accumulation=2 để effective batch vẫn là 64.

In [ ]:
CACHED_MODEL = CACHE / 'qa_reranker_full_e1.zip'
if not (FT_MODEL / 'config.json').exists():
    if CACHED_MODEL.exists():
        subprocess.run(['unzip', '-q', '-o', str(CACHED_MODEL), '-d', str(WORK)], check=True)
    else:
        accum = 2 if TRAIN_BATCH == 32 else 1
        subprocess.run([
            sys.executable, '-u', str(trainer), '--base', BASE_MODEL,
            '--data', str(PAIRS), '--out', str(FT_MODEL),
            '--log', str(WORK / 'train_qa_reranker_full_e1.log'),
            '--full', '--precision', 'bf16', '--no-checkpointing',
            '--epochs', '1', '--batch', str(TRAIN_BATCH), '--accum', str(accum),
            '--neg', '8', '--maxlen', '384', '--lr', '2e-5',
            '--warmup', '0.08', '--workers', '4'
        ], cwd=str(ROOT / 'Finetune-LegalIR'), check=True)
        subprocess.run(['zip', '-0qr', str(CACHED_MODEL), FT_MODEL.name],
                       cwd=str(WORK), check=True)
        shutil.copy2(WORK / 'train_qa_reranker_full_e1.log', CACHE / 'train_qa_reranker_full_e1.log')
print('MODEL READY:', FT_MODEL)

## Đánh giá epoch 1 và sweep beta
So sánh trên đúng block `offset=200, n=200`; mốc zero-shot trước đó là METEOR 0.5347.

In [ ]:
qa_predict = ROOT / 'Retrieval-LegalIR/qa_predict.py'
for beta in (0.4, 0.5, 0.6, 0.7, 0.8):
    print(f'\n========== FULL FT | BETA={beta} ==========')
    subprocess.run([
        sys.executable, '-u', str(qa_predict),
        '--qa-dir', str(QA_DIR), '--index', str(index_dir), '--emb', str(emb_dir),
        '--retriever', 'hybrid', '--alpha', '0.7',
        '--reranker', str(FT_MODEL), '--beta', str(beta), '--batch', '128',
        '--eval', '--offset', '200', '-n', '200',
        '--nchunk', '3', '--style', 'cite'
    ], cwd=str(ROOT / 'Retrieval-LegalIR'), check=True)

## Optional: epoch 2
Chỉ bật nếu epoch 1 đã thắng 0.5347. Epoch 2 dùng learning rate thấp hơn và lưu model riêng để không ghi đè epoch 1.

In [ ]:
RUN_EPOCH_2 = False
FT_MODEL_E2 = WORK / 'qa_reranker_full_e2'
if RUN_EPOCH_2:
    accum = 2 if TRAIN_BATCH == 32 else 1
    subprocess.run([
        sys.executable, '-u', str(trainer), '--base', str(FT_MODEL),
        '--data', str(PAIRS), '--out', str(FT_MODEL_E2),
        '--log', str(WORK / 'train_qa_reranker_full_e2.log'),
        '--full', '--precision', 'bf16', '--no-checkpointing',
        '--epochs', '1', '--batch', str(TRAIN_BATCH), '--accum', str(accum),
        '--neg', '8', '--maxlen', '384', '--lr', '8e-6',
        '--warmup', '0.03', '--workers', '4'
    ], cwd=str(ROOT / 'Finetune-LegalIR'), check=True)
    archive = CACHE / 'qa_reranker_full_e2.zip'
    subprocess.run(['zip', '-0qr', str(archive), FT_MODEL_E2.name],
                   cwd=str(WORK), check=True)
    print('EPOCH 2 READY:', FT_MODEL_E2)

## Sinh submission
Điền beta tốt nhất từ cell sweep rồi chạy.

In [ ]:
BEST_BETA = 0.6
BEST_MODEL = FT_MODEL
submission = WORK / 'submission_qa_full_reranker'
subprocess.run([
    sys.executable, '-u', str(qa_predict),
    '--qa-dir', str(QA_DIR), '--questions', str(QA_DIR / 'public-official.json'),
    '--index', str(index_dir), '--emb', str(emb_dir),
    '--retriever', 'hybrid', '--alpha', '0.7',
    '--reranker', str(BEST_MODEL), '--beta', str(BEST_BETA), '--batch', '128',
    '--nchunk', '3', '--style', 'cite', '--out', str(submission)
], cwd=str(ROOT / 'Retrieval-LegalIR'), check=True)
shutil.copy2(str(submission) + '.zip', CACHE / (submission.name + '.zip'))
print('SUBMISSION:', str(submission) + '.zip')